# AdoTHC — workflow automatizado de snRNA-seq de hipocampo

**Desenho fixado:** 8 hipocampos inteiros de camundongos, quatro VEH e quatro THC; um animal por amostra; tecido não perfundido. A unidade experimental de THC versus VEH é o **animal**, nunca o núcleo.

Este notebook lê exclusivamente as matrizes **CellBender filtradas e produzidas com parâmetros uniformes**, executa QC por amostra, Scrublet por biblioteca, análise integrada sem alterar as contagens, Leiden amplo e detalhado, UMAP, t-SNE estratificado, anotação assistida com incerteza explícita, composição por animal, pseudobulk, PyDESeq2, GO-BP e GSEA.

### Decisões metodológicas

- `counts` preserva inteiros corrigidos pelo CellBender e é a única camada usada no pseudobulk.
- HVGs são selecionados considerando `sample`; genes técnicos não dirigem PCA.
- Não se regressam UMI, genes, mitocondriais ou ribossomais.
- A correção Harmony é apenas uma sensibilidade opcional e nunca entra em DE.
- O cluster amplo serve para composição e DE com mais poder. O cluster detalhado serve para subtipos e hipóteses.
- Rótulos automáticos são candidatos; baixa confiança permanece `Unannotated`.
- Resultados são gravados em etapas. Se o kernel parar, ajuste `START_STAGE` e recomece do checkpoint indicado.

**Uso:** edite somente a célula CONFIGURAÇÃO, escolha `Run > Run All Cells` e acompanhe as mensagens `STAGE OK`.

In [ ]:
# ========================== CONFIGURAÇÃO ==========================
from pathlib import Path

DATA_ROOT = Path(r"C:\Users\admin\Desktop\Results_transcriptomics\snRNAseq_analysis\Result_X202SC26075533-Z01-F001_GRCm39_2024-A\1.Data_process")

# Coloque aqui os 8 arquivos finais uniformes *_lr5e5_filtered.h5.
CELLBENDER_ROOT = DATA_ROOT / "AdoTHC_cellbender_final_inputs"
OUTPUT_ROOT = DATA_ROOT / "AdoTHC_scanpy_automated_results"

SAMPLES = ["VEH1", "VEH2", "VEH3", "VEH4", "THC1", "THC2", "THC3", "THC4"]
GROUP = {s: ("VEH" if s.startswith("VEH") else "THC") for s in SAMPLES}
RANDOM_SEED = 20260924
N_CPUS = 4

# Segurança contra mistura da primeira rodada e da rodada learning-rate 5e-5.
REQUIRE_UNIFORM_LR5E5 = True

# QC de núcleos: piso predefinido + teto robusto específico de cada amostra.
MIN_GENES = 500
MIN_COUNTS = 1000
MAX_PCT_MT = 5.0
UPPER_MAD = 5.0
REMOVE_PREDICTED_DOUBLETS = True
EXPECTED_DOUBLET_RATE = 0.06

# Representação e clusters.
N_TOP_HVG = 3000
N_PCS = 40
N_NEIGHBORS = 15
BROAD_RESOLUTION = 0.35
FINE_RESOLUTION = 0.80
RESOLUTION_SWEEP = [0.20, 0.35, 0.50, 0.65, 0.80, 1.00, 1.20]
USE_HARMONY = False  # sensibilidade; requer harmonypy. Contagens de DE nunca são integradas.
RUN_TSNE = True
TSNE_MAX_NUCLEI = 40000

# Anotação. “Probabilidades” são escores relativos, não probabilidades calibradas.
AUTO_LABEL_MIN_PROB = 0.45
AUTO_LABEL_MIN_MARGIN = 0.08

# Pseudobulk animal × tipo amplo.
MIN_NUCLEI_PER_SAMPLE_TYPE = 50
MIN_SAMPLES_PER_GROUP = 3
MIN_TOTAL_PB_COUNTS = 20000
MIN_CPM = 1.0
MIN_SAMPLES_GENE = 3
N_BOOTSTRAP = 5000
RUN_PATHWAYS = True  # requer internet para Enrichr/MSigDB
RUN_DE_LOO = False

# Retomada: 0 = do início; 1 = após QC; 2 = após clusters; 3 = após anotação.
START_STAGE = 0
# ================================================================

assert set(SAMPLES) == set(GROUP)
assert 0 <= START_STAGE <= 3

## 0. Ambiente e pastas

In [ ]:
import sys, re, json, platform, warnings, hashlib, itertools
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import median_abs_deviation
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad

if sys.version_info >= (3, 14):
    raise RuntimeError("Selecione o kernel Python (scanpy_thc) 3.12; Python 3.14 não é o ambiente validado.")

required = ["scanpy", "anndata", "igraph", "leidenalg", "pydeseq2", "gseapy", "openpyxl", "adjustText", "scikit-misc"]
missing = []
for package in required:
    try: version(package)
    except PackageNotFoundError: missing.append(package)
if missing:
    raise RuntimeError("Pacotes ausentes: " + ", ".join(missing) + ". Rode no terminal scanpy_thc: pip install " + " ".join(missing))
if USE_HARMONY:
    try: version("harmonypy")
    except PackageNotFoundError: raise RuntimeError("USE_HARMONY=True requer: pip install harmonypy")

np.random.seed(RANDOM_SEED)
sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, dpi_save=300, frameon=False, fontsize=10)
sns.set_theme(style="white", context="paper")

DIRS = {name: OUTPUT_ROOT / name for name in [
    "checkpoints", "qc", "embeddings", "markers", "annotation", "composition",
    "pseudobulk", "de", "pathways", "reports", "logs"
]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0], "| Scanpy:", version("scanpy"), "| AnnData:", version("anndata"))
print("Entrada:", CELLBENDER_ROOT)
print("Saída:", OUTPUT_ROOT)

## 1. Manifesto e validação dos 8 arquivos CellBender

In [ ]:
def sha256_prefix(path, mib=4):
    h = hashlib.sha256()
    with open(path, "rb") as f: h.update(f.read(mib * 1024**2))
    return h.hexdigest()

def find_one_cellbender_file(sample):
    patterns = [f"**/{sample}*_lr5e5_filtered.h5"] if REQUIRE_UNIFORM_LR5E5 else [f"**/{sample}*_filtered.h5"]
    hits = sorted({p.resolve() for pat in patterns for p in CELLBENDER_ROOT.glob(pat) if p.is_file()})
    # Excluir posterior e qualquer matriz sem o sufixo filtered.
    hits = [p for p in hits if "posterior" not in p.name.lower() and p.name.lower().endswith("_filtered.h5")]
    if len(hits) != 1:
        raise FileNotFoundError(
            f"{sample}: esperava exatamente 1 arquivo final em {CELLBENDER_ROOT}; encontrei {len(hits)}: {hits}. "
            "Não misture arquivos fpr0 antigos com os uniformes lr5e5."
        )
    return hits[0]

rows = []
for sample in SAMPLES:
    p = find_one_cellbender_file(sample)
    rows.append({"sample": sample, "group": GROUP[sample], "input": str(p),
                 "size_MiB": round(p.stat().st_size/1024**2, 2), "sha256_first4MiB": sha256_prefix(p)})
input_manifest = pd.DataFrame(rows)
assert input_manifest["input"].nunique() == 8
display(input_manifest)
input_manifest.to_csv(DIRS["logs"] / "input_manifest.csv", index=False)
print("PREFLIGHT OK — oito arquivos únicos e uniformes.")

## 2. QC por amostra e Scrublet por biblioteca

In [ ]:
def read_10x_h5_clean(path):
    x = sc.read_10x_h5(path)
    x.var_names_make_unique()
    x.X = x.X.tocsr().astype(np.float32)
    return x

def robust_upper(values, nmads=5.0):
    z = np.log1p(np.asarray(values, dtype=float))
    med = np.median(z); mad = median_abs_deviation(z, scale="normal")
    return float(np.max(values) if mad == 0 else np.expm1(med + nmads * mad))

def add_gene_flags(x):
    names = pd.Index(x.var_names.astype(str))
    x.var["mt"] = names.str.lower().str.startswith("mt-")
    x.var["ribo"] = names.str.match(r"^Rp[sl]")
    x.var["hb"] = names.str.match(r"^Hb[ab]")
    x.var["ambient_myelin"] = names.isin(["Mbp", "Plp1", "Mobp", "Mog", "Mag"])

def qc_one(sample, path):
    x = read_10x_h5_clean(path)
    x.obs_names = pd.Index([f"{sample}:{b}" for b in x.obs_names])
    x.obs["sample"] = sample; x.obs["group"] = GROUP[sample]
    add_gene_flags(x)
    sc.pp.calculate_qc_metrics(x, qc_vars=["mt", "ribo", "hb", "ambient_myelin"], percent_top=None, log1p=False, inplace=True)
    ug = robust_upper(x.obs["n_genes_by_counts"], UPPER_MAD)
    uc = robust_upper(x.obs["total_counts"], UPPER_MAD)
    x.obs["pass_basic_qc"] = ((x.obs.n_genes_by_counts >= MIN_GENES) & (x.obs.total_counts >= MIN_COUNTS) &
                              (x.obs.pct_counts_mt <= MAX_PCT_MT) & (x.obs.n_genes_by_counts <= ug) &
                              (x.obs.total_counts <= uc))
    x.obs["predicted_doublet"] = False; x.obs["doublet_score"] = np.nan
    xq = x[x.obs.pass_basic_qc].copy()
    scrublet_status = "not_run"
    if xq.n_obs > 500:
        try:
            sc.pp.scrublet(xq, expected_doublet_rate=EXPECTED_DOUBLET_RATE, random_state=RANDOM_SEED)
            x.obs.loc[xq.obs_names, "predicted_doublet"] = xq.obs.predicted_doublet.astype(bool)
            x.obs.loc[xq.obs_names, "doublet_score"] = xq.obs.doublet_score.astype(float)
            scrublet_status = "ok"
        except Exception as exc:
            scrublet_status = f"failed: {exc}"
            warnings.warn(f"Scrublet falhou em {sample}; o workflow não removerá doublets desta amostra: {exc}")
    keep = x.obs.pass_basic_qc.copy()
    if REMOVE_PREDICTED_DOUBLETS and scrublet_status == "ok": keep &= ~x.obs.predicted_doublet
    row = {"sample":sample, "group":GROUP[sample], "input_nuclei":x.n_obs, "upper_genes":ug, "upper_counts":uc,
           "pass_basic_qc":int(x.obs.pass_basic_qc.sum()), "predicted_doublets":int(x.obs.predicted_doublet.sum()),
           "retained":int(keep.sum()), "retained_fraction":float(keep.mean()), "scrublet_status":scrublet_status,
           "median_genes":float(x.obs.n_genes_by_counts.median()), "median_counts":float(x.obs.total_counts.median()),
           "median_pct_mt":float(x.obs.pct_counts_mt.median()), "median_pct_hb":float(x.obs.pct_counts_hb.median()),
           "median_pct_myelin":float(x.obs.pct_counts_ambient_myelin.median())}
    out_obs = x.obs.copy(); out_obs.insert(0, "barcode", out_obs.index)
    out_obs.to_csv(DIRS["qc"] / f"{sample}_nucleus_qc.csv", index=False)
    return x[keep].copy(), row

CK1 = DIRS["checkpoints"] / "01_qc_counts.h5ad"
if START_STAGE >= 1:
    adata = sc.read_h5ad(CK1)
    qc_summary = pd.read_csv(DIRS["qc"] / "qc_summary_by_sample.csv")
    print("Checkpoint QC carregado:", CK1)
else:
    adatas, qc_rows = {}, []
    for r in input_manifest.itertuples(index=False):
        print("QC:", r.sample)
        adatas[r.sample], q = qc_one(r.sample, r.input); qc_rows.append(q)
    qc_summary = pd.DataFrame(qc_rows)
    qc_summary.to_csv(DIRS["qc"] / "qc_summary_by_sample.csv", index=False)
    adata = ad.concat(adatas, join="inner", merge="same")
    adata.obs["sample"] = pd.Categorical(adata.obs["sample"], SAMPLES, ordered=True)
    adata.obs["group"] = pd.Categorical(adata.obs["group"], ["VEH", "THC"], ordered=True)
    adata.layers["counts"] = adata.X.copy().tocsr()
    assert set(adata.obs["sample"].astype(str).unique()) == set(SAMPLES)
    assert np.allclose(adata.layers["counts"].data, np.rint(adata.layers["counts"].data))
    adata.write_h5ad(CK1, compression="gzip")
display(qc_summary)
print("STAGE 1 OK —", adata.n_obs, "núcleos;", adata.n_vars, "genes.")

In [ ]:
qc_long = pd.concat([pd.read_csv(DIRS["qc"] / f"{s}_nucleus_qc.csv").assign(sample=s, group=GROUP[s]) for s in SAMPLES], ignore_index=True)
vars_qc = ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb", "pct_counts_ambient_myelin", "doublet_score"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, var in zip(axes.flat, vars_qc):
    sns.violinplot(data=qc_long, x="sample", y=var, hue="group", dodge=False, cut=0, inner="quartile", ax=ax)
    ax.tick_params(axis="x", rotation=45)
    if ax.legend_: ax.legend_.remove()
fig.tight_layout(); fig.savefig(DIRS["qc"] / "QC_distributions_before_filtering.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(8,4)); sns.barplot(data=qc_summary, x="sample", y="retained_fraction", hue="group", dodge=False, ax=ax)
ax.tick_params(axis="x", rotation=45); ax.set_ylim(0,1); fig.tight_layout()
fig.savefig(DIRS["qc"] / "QC_retained_fraction.png", bbox_inches="tight"); plt.show()

## 3. Normalização, HVGs, PCA, UMAP e dois níveis de Leiden

In [ ]:
CK2 = DIRS["checkpoints"] / "02_clustered.h5ad"
if START_STAGE >= 2:
    adata = sc.read_h5ad(CK2)
    print("Checkpoint de clusters carregado:", CK2)
else:
    sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3", n_top_genes=N_TOP_HVG, batch_key="sample", subset=False)
    technical = adata.var[["mt","ribo","hb","ambient_myelin"]].fillna(False).any(axis=1)
    adata.var["use_for_pca"] = adata.var.highly_variable & ~technical
    sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)
    # PCA esparsa: evita criar uma matriz densa núcleos × genes no PC de 32 GB.
    sc.tl.pca(adata, n_comps=max(N_PCS, 50), mask_var="use_for_pca", zero_center=True,
              svd_solver="arpack", random_state=RANDOM_SEED)
    rep = "X_pca"
    if USE_HARMONY:
        import scanpy.external as sce
        sce.pp.harmony_integrate(adata, key="sample", basis="X_pca", adjusted_basis="X_pca_harmony", random_state=RANDOM_SEED)
        rep = "X_pca_harmony"
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, use_rep=rep, random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED, min_dist=0.3)
    for res in RESOLUTION_SWEEP:
        sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res:.2f}", flavor="igraph",
                     n_iterations=2, directed=False, random_state=RANDOM_SEED)
    adata.obs["cluster_broad"] = adata.obs[f"leiden_{BROAD_RESOLUTION:.2f}"].astype("category")
    adata.obs["cluster_fine"] = adata.obs[f"leiden_{FINE_RESOLUTION:.2f}"].astype("category")
    adata.uns["analysis_representation"] = rep
    adata.write_h5ad(CK2, compression="gzip")

resolution_rows = []
from sklearn.metrics import adjusted_rand_score
for i, res in enumerate(RESOLUTION_SWEEP):
    key=f"leiden_{res:.2f}"; sizes=adata.obs[key].value_counts()
    resolution_rows.append({"resolution":res, "n_clusters":len(sizes), "smallest_cluster":int(sizes.min()),
                            "median_cluster":float(sizes.median()), "ARI_vs_previous":np.nan if i==0 else adjusted_rand_score(adata.obs[f"leiden_{RESOLUTION_SWEEP[i-1]:.2f}"], adata.obs[key])})
resolution_diagnostics = pd.DataFrame(resolution_rows)
resolution_diagnostics.to_csv(DIRS["markers"] / "resolution_diagnostics.csv", index=False)
display(resolution_diagnostics)

for color, name in [(["cluster_broad"],"UMAP_broad_clusters"), (["cluster_fine"],"UMAP_fine_clusters"),
                    (["sample"],"UMAP_samples"), (["group"],"UMAP_groups"),
                    (["total_counts","n_genes_by_counts","pct_counts_mt"],"UMAP_QC")]:
    sc.pl.umap(adata, color=color, ncols=min(3,len(color)), show=False, legend_loc="on data" if "cluster" in name else "right margin")
    plt.savefig(DIRS["embeddings"] / f"{name}.png", bbox_inches="tight"); plt.show()
print("STAGE 2 OK — representação:", adata.uns["analysis_representation"])

In [ ]:
# t-SNE é calculado em subconjunto estratificado para controlar RAM/tempo; UMAP contém todos os núcleos.
if RUN_TSNE:
    rng = np.random.default_rng(RANDOM_SEED)
    per_stratum = max(50, TSNE_MAX_NUCLEI // max(1, adata.obs.groupby(["sample","cluster_fine"], observed=True).ngroups))
    chosen=[]
    for _, idx in adata.obs.groupby(["sample","cluster_fine"], observed=True).indices.items():
        idx=np.asarray(idx); chosen.extend(rng.choice(idx, min(len(idx), per_stratum), replace=False))
    chosen=np.asarray(sorted(set(chosen)))
    if len(chosen)>TSNE_MAX_NUCLEI: chosen=rng.choice(chosen, TSNE_MAX_NUCLEI, replace=False)
    ats=adata[chosen].copy(); sc.tl.tsne(ats, use_rep=adata.uns["analysis_representation"], n_pcs=N_PCS, random_state=RANDOM_SEED, n_jobs=N_CPUS)
    for color,name in [(["cluster_fine"],"tSNE_fine_clusters_subset"),(["sample"],"tSNE_samples_subset"),(["group"],"tSNE_groups_subset")]:
        sc.pl.tsne(ats,color=color,show=False,legend_loc="on data" if "cluster" in name else "right margin")
        plt.savefig(DIRS["embeddings"] / f"{name}.png",bbox_inches="tight"); plt.show()
    ats.obs.assign(tsne1=ats.obsm["X_tsne"][:,0],tsne2=ats.obsm["X_tsne"][:,1]).to_csv(DIRS["embeddings"] / "tSNE_stratified_subset_coordinates.csv")
    print("t-SNE:", ats.n_obs, "núcleos estratificados.")

## 4. Marcadores e anotação ampla com confiança explícita

In [ ]:
MARKER_MODULES = {
 "Excitatory_neuron":["Slc17a7","Camk2a","Satb2","Tbr1","Neurod6","Rorb"],
 "Inhibitory_neuron":["Gad1","Gad2","Slc6a1","Dlx1","Dlx2","Slc32a1"],
 "Astrocyte":["Aqp4","Aldoc","Slc1a3","Gja1","Sox9","Slc1a2"],
 "Oligodendrocyte":["Mog","Mag","Cnp","Sox10","Olig2","Myrf"],
 "OPC":["Pdgfra","Cspg4","Olig1","Ptprz1","Vcan"],
 "Microglia":["P2ry12","Tmem119","Cx3cr1","C1qa","Hexb","Sall1"],
 "Peripheral_immune":["Ptprc","Lsp1","Fcer1g","Tyrobp","Lst1","S100a8"],
 "Endothelial":["Cldn5","Pecam1","Kdr","Esam","Slco1a4"],
 "Pericyte_VSMC":["Pdgfrb","Rgs5","Des","Acta2","Cspg4","Kcnj8"],
 "Ependymal":["Foxj1","Pifo","Dnah12","Tmem212","Dynlrb2"],
 "Choroid_plexus":["Ttr","Klotho","Otx2","Aqp1","Folr1"]}

def stratified_marker_subset(a, groupby, max_per_cluster=4000):
    rng=np.random.default_rng(RANDOM_SEED); chosen=[]
    for _,idx in a.obs.groupby(groupby,observed=True).indices.items():
        idx=np.asarray(idx); chosen.extend(rng.choice(idx,min(len(idx),max_per_cluster),replace=False))
    return a[np.asarray(chosen)].copy()

def annotation_table(a, cluster_key):
    score_cols={}
    for label,genes in MARKER_MODULES.items():
        present=[g for g in genes if g in a.var_names]
        if len(present)>=2:
            col=f"score__{label}"; sc.tl.score_genes(a,present,score_name=col,random_state=RANDOM_SEED); score_cols[label]=col
    means=a.obs.groupby(cluster_key,observed=True)[list(score_cols.values())].mean(); means.columns=list(score_cols)
    z=(means-means.mean())/means.std().replace(0,1); ez=np.exp(z.sub(z.max(axis=1),axis=0)); probs=ez.div(ez.sum(axis=1),axis=0)
    order=np.argsort(-probs.to_numpy(),axis=1); labels=probs.columns.to_numpy(); arr=probs.to_numpy(); ii=np.arange(len(probs))
    out=pd.DataFrame({"cluster":probs.index.astype(str),"candidate_1":labels[order[:,0]],"probability_1":arr[ii,order[:,0]],
                      "candidate_2":labels[order[:,1]],"probability_2":arr[ii,order[:,1]]})
    out["margin"]=out.probability_1-out.probability_2
    out["auto_label"]=np.where((out.probability_1>=AUTO_LABEL_MIN_PROB)&(out.margin>=AUTO_LABEL_MIN_MARGIN),out.candidate_1,"Unannotated")
    return out.merge(probs.reset_index().rename(columns={cluster_key:"cluster", "index":"cluster"}),on="cluster",how="left")

CK3=DIRS["checkpoints"] / "03_annotated.h5ad"
if START_STAGE>=3:
    adata=sc.read_h5ad(CK3); annotation=pd.read_csv(DIRS["annotation"] / "broad_cluster_annotation_probabilities.csv")
else:
    marker_data=stratified_marker_subset(adata,"cluster_fine")
    sc.tl.rank_genes_groups(marker_data,groupby="cluster_fine",method="wilcoxon",pts=True,corr_method="benjamini-hochberg",key_added="fine_markers")
    markers=sc.get.rank_genes_groups_df(marker_data,group=None,key="fine_markers")
    markers.to_csv(DIRS["markers"] / "all_fine_cluster_markers.csv",index=False)
    annotation=annotation_table(adata,"cluster_broad")
    # Edite somente se a revisão de marcadores justificar. IDs dependem desta execução.
    MANUAL_OVERRIDES = {}
    annotation["final_label"]=[MANUAL_OVERRIDES.get(str(c),a) for c,a in zip(annotation.cluster,annotation.auto_label)]
    annotation.to_csv(DIRS["annotation"] / "broad_cluster_annotation_probabilities.csv",index=False)
    label_map=dict(zip(annotation.cluster.astype(str),annotation.final_label))
    adata.obs["cell_type_broad"]=adata.obs.cluster_broad.astype(str).map(label_map).astype("category")
    fine_candidates=annotation_table(adata,"cluster_fine")
    fine_candidates.to_csv(DIRS["annotation"] / "fine_cluster_candidate_probabilities.csv",index=False)
    adata.obs["cell_type_fine"]="Cluster_"+adata.obs.cluster_fine.astype(str)
    adata.obs["cell_type_fine"]=adata.obs["cell_type_fine"].astype("category")
    adata.write_h5ad(CK3,compression="gzip")
display(annotation)
sc.pl.dotplot(adata,{k:[g for g in v if g in adata.var_names] for k,v in MARKER_MODULES.items()},groupby="cluster_broad",standard_scale="var",show=False)
plt.savefig(DIRS["annotation"] / "broad_marker_dotplot.png",bbox_inches="tight"); plt.show()
sc.pl.umap(adata,color="cell_type_broad",show=False); plt.savefig(DIRS["annotation"] / "UMAP_broad_cell_types.png",bbox_inches="tight"); plt.show()
print("STAGE 3 OK — rótulos incertos permaneceram Unannotated.")

In [ ]:
# Controle de dominância por amostra e vazamento residual de marcadores após CellBender.
cluster_sample=pd.crosstab(adata.obs.cluster_fine,adata.obs["sample"])
cluster_sample["max_sample_fraction"]=cluster_sample.max(axis=1)/cluster_sample.sum(axis=1)
cluster_sample["flag_sample_dominance_gt60pct"]=cluster_sample.max_sample_fraction>0.60
cluster_sample.to_csv(DIRS["markers"] / "fine_clusters_by_sample.csv")

diag_genes=[g for g in ["Mbp","Plp1","Mobp","Slc17a7","Gad1","Gad2","Ttr","Hba-a1","Hbb-bs","Ptprc","P2ry12"] if g in adata.var_names]
rows=[]; counts=adata.layers["counts"].tocsr()
for (s,ct),idx in adata.obs.groupby(["sample","cell_type_broad"],observed=True).indices.items():
    idx=np.asarray(idx)
    for g in diag_genes:
        j=adata.var_names.get_loc(g); pos=int((counts[idx,j]>0).sum())
        rows.append({"sample":str(s),"group":GROUP[str(s)],"cell_type_broad":str(ct),"gene":g,"n_nuclei":len(idx),"n_positive":pos,"fraction_positive":pos/len(idx)})
ambient_diagnostic=pd.DataFrame(rows)
ambient_diagnostic.to_csv(DIRS["qc"] / "post_CellBender_marker_leakage.csv",index=False)
display(cluster_sample.sort_values("max_sample_fraction",ascending=False).head(15))

## 5. Composição por animal: bootstrap e permutação exata

In [ ]:
composition=adata.obs.groupby(["sample","group","cell_type_broad"],observed=True).size().rename("n_nuclei").reset_index()
totals=adata.obs.groupby("sample",observed=True).size().rename("total_nuclei")
composition=composition.join(totals,on="sample"); composition["fraction"]=composition.n_nuclei/composition.total_nuclei
composition.to_csv(DIRS["composition"] / "broad_composition_by_animal.csv",index=False)

fig,ax=plt.subplots(figsize=(12,5)); sns.barplot(data=composition,x="cell_type_broad",y="fraction",hue="group",errorbar="se",capsize=.15,ax=ax)
sns.stripplot(data=composition,x="cell_type_broad",y="fraction",hue="group",dodge=True,color="black",size=4,ax=ax,legend=False)
ax.tick_params(axis="x",rotation=45); fig.tight_layout(); fig.savefig(DIRS["composition"] / "composition_group_animal_points.png",bbox_inches="tight"); plt.show()

def composition_inference(df):
    rng=np.random.default_rng(RANDOM_SEED); out=[]; assignments=list(itertools.combinations(SAMPLES,4))
    for ct,d in df.groupby("cell_type_broad",observed=True):
        vals=d.set_index("sample").reindex(SAMPLES).fraction.fillna(0)
        observed=vals[[s for s in SAMPLES if GROUP[s]=="THC"]].mean()-vals[[s for s in SAMPLES if GROUP[s]=="VEH"]].mean()
        perm=[]
        for thc in assignments:
            veh=[s for s in SAMPLES if s not in thc]; perm.append(vals[list(thc)].mean()-vals[veh].mean())
        boots=[]; vehv=vals[[s for s in SAMPLES if GROUP[s]=="VEH"]].to_numpy(); thcv=vals[[s for s in SAMPLES if GROUP[s]=="THC"]].to_numpy()
        for _ in range(N_BOOTSTRAP): boots.append(rng.choice(thcv,4,True).mean()-rng.choice(vehv,4,True).mean())
        out.append({"cell_type_broad":str(ct),"THC_minus_VEH":observed,"bootstrap_CI_low":np.quantile(boots,.025),
                    "bootstrap_CI_high":np.quantile(boots,.975),"exact_permutation_p":(np.abs(perm)>=abs(observed)-1e-15).mean()})
    ans=pd.DataFrame(out)
    from statsmodels.stats.multitest import multipletests
    ans["permutation_FDR_BH"]=multipletests(ans.exact_permutation_p,method="fdr_bh")[1]
    return ans
composition_stats=composition_inference(composition)
composition_stats.to_csv(DIRS["composition"] / "composition_animal_inference.csv",index=False); display(composition_stats)

## 6. Pseudobulk: soma de contagens por animal × tipo amplo

In [ ]:
def sum_rows(matrix,rows): return np.asarray(matrix[rows].sum(axis=0)).ravel()
pb_rows=[]; pb_counts=[]; counts=adata.layers["counts"].tocsr()
for (sample,group,ct),idx in adata.obs.groupby(["sample","group","cell_type_broad"],observed=True).indices.items():
    idx=np.asarray(idx); values=sum_rows(counts,idx)
    pb_rows.append({"pb_id":f"{sample}__{ct}","sample":str(sample),"group":str(group),"cell_type_broad":str(ct),"n_nuclei":len(idx),"library_size":int(values.sum())})
    pb_counts.append(values)
pb_meta=pd.DataFrame(pb_rows).set_index("pb_id")
pb_matrix=pd.DataFrame(np.vstack(pb_counts).astype(np.int64),index=pb_meta.index,columns=adata.var_names)
qual=(pb_meta.n_nuclei>=MIN_NUCLEI_PER_SAMPLE_TYPE)&(pb_meta.library_size>=MIN_TOTAL_PB_COUNTS)
elig=[]
for ct,d in pb_meta.assign(qualifies=qual).groupby("cell_type_broad"):
    nv=int(d.query("group=='VEH'").qualifies.sum()); nt=int(d.query("group=='THC'").qualifies.sum())
    elig.append({"cell_type_broad":ct,"qualifying_VEH":nv,"qualifying_THC":nt,"eligible":nv>=MIN_SAMPLES_PER_GROUP and nt>=MIN_SAMPLES_PER_GROUP and ct!="Unannotated"})
eligibility=pd.DataFrame(elig)
pb_meta.to_csv(DIRS["pseudobulk"] / "pseudobulk_metadata.csv"); pb_matrix.to_csv(DIRS["pseudobulk"] / "pseudobulk_counts.csv")
eligibility.to_csv(DIRS["pseudobulk"] / "pseudobulk_eligibility.csv",index=False); display(eligibility)

## 7. DE animal-level com PyDESeq2

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def run_de(ct,excluded_sample=None):
    meta=pb_meta[pb_meta.cell_type_broad.eq(ct)].copy()
    keep=(meta.n_nuclei>=MIN_NUCLEI_PER_SAMPLE_TYPE)&(meta.library_size>=MIN_TOTAL_PB_COUNTS)
    if excluded_sample: keep &= meta["sample"].ne(excluded_sample)
    meta=meta.loc[keep].copy(); c=pb_matrix.loc[meta.index].copy()
    cpm=c.div(c.sum(axis=1),axis=0)*1e6; c=c.loc[:,(cpm>=MIN_CPM).sum(axis=0)>=MIN_SAMPLES_GENE].astype(int)
    meta["group"]=pd.Categorical(meta.group,["VEH","THC"],ordered=True)
    if meta.groupby("group",observed=True).size().min()<2: raise ValueError("menos de 2 animais em um grupo")
    try: dds=DeseqDataSet(counts=c,metadata=meta,design="~group",refit_cooks=True,n_cpus=N_CPUS)
    except TypeError: dds=DeseqDataSet(counts=c,metadata=meta,design_factors="group",refit_cooks=True,n_cpus=N_CPUS)
    dds.deseq2(); stat=DeseqStats(dds,contrast=["group","THC","VEH"],alpha=.05,n_cpus=N_CPUS); stat.summary()
    r=stat.results_df.reset_index().rename(columns={stat.results_df.index.name or "index":"gene"})
    r["cell_type_broad"]=ct; r["contrast"]="THC_vs_VEH"; r["n_animals"]=len(meta); r["excluded_sample"]=excluded_sample
    return r,meta,c

de_results={}
for ct in eligibility.query("eligible").cell_type_broad:
    print("DE:",ct)
    try:
        r,m,c=run_de(ct); de_results[ct]=r; safe=re.sub(r"[^A-Za-z0-9_.-]+","_",ct)
        r.to_csv(DIRS["de"] / f"DE_{safe}_THC_vs_VEH.csv",index=False); c.to_csv(DIRS["pseudobulk"] / f"edgeR_counts_{safe}.csv"); m.to_csv(DIRS["pseudobulk"] / f"edgeR_metadata_{safe}.csv")
    except Exception as exc: warnings.warn(f"DE falhou para {ct}: {exc}")
de_all=pd.concat(de_results.values(),ignore_index=True) if de_results else pd.DataFrame()
if not de_all.empty: de_all.to_csv(DIRS["de"] / "DE_all_broad_cell_types.csv",index=False)
display(de_all.groupby("cell_type_broad").agg(genes=("gene","size"),FDR05=("padj",lambda x:int((x<.05).sum()))) if not de_all.empty else de_all)

In [ ]:
from adjustText import adjust_text
def volcano(d,title,path):
    x=d.dropna(subset=["log2FoldChange","padj"]).copy(); x["minus_log10_FDR"]=-np.log10(x.padj.clip(lower=1e-300)); x["status"]="NS"
    x.loc[(x.padj<.05)&(x.log2FoldChange>=.5),"status"]="Higher THC"; x.loc[(x.padj<.05)&(x.log2FoldChange<=-.5),"status"]="Lower THC"
    fig,ax=plt.subplots(figsize=(6,5)); sns.scatterplot(data=x,x="log2FoldChange",y="minus_log10_FDR",hue="status",palette={"NS":"#BDBDBD","Higher THC":"#D55E00","Lower THC":"#0072B2"},s=18,linewidth=0,ax=ax)
    ax.axhline(-np.log10(.05),color="black",ls="--",lw=.8); ax.axvline(0,color="black",lw=.6)
    top=x.assign(abs_lfc=x.log2FoldChange.abs()).sort_values(["padj","abs_lfc"],ascending=[True,False]).head(12)
    texts=[ax.text(r.log2FoldChange,r.minus_log10_FDR,r.gene,fontsize=7) for r in top.itertuples()]; adjust_text(texts,ax=ax,arrowprops=dict(arrowstyle="-",color="grey",lw=.5))
    ax.set_title(title); fig.tight_layout(); fig.savefig(path,bbox_inches="tight"); plt.show()
for ct,r in de_results.items():
    safe=re.sub(r"[^A-Za-z0-9_.-]+","_",ct); volcano(r,f"{ct}: THC vs VEH",DIRS["de"] / f"volcano_{safe}.png")

## 8. GO-BP e GSEA preranked

In [ ]:
import gseapy as gp
go_all=[]; gsea_all=[]
if RUN_PATHWAYS:
    try:
        libs=gp.get_library_name(organism="Mouse"); candidates=[x for x in libs if "GO_Biological_Process" in x]
        GO_LIBRARY=sorted(candidates)[-1] if candidates else "GO_Biological_Process_2023"
        for ct,r in de_results.items():
            safe=re.sub(r"[^A-Za-z0-9_.-]+","_",ct); d=r.dropna(subset=["stat","log2FoldChange"]).copy()
            for direction,genes in {"higher_THC":d.query("padj<.05 and log2FoldChange>.5").gene.tolist(),"lower_THC":d.query("padj<.05 and log2FoldChange<-.5").gene.tolist()}.items():
                if len(genes)>=10:
                    tab=gp.enrichr(gene_list=genes,gene_sets=GO_LIBRARY,organism="Mouse",outdir=None).results
                    tab["cell_type_broad"]=ct; tab["direction"]=direction; tab.to_csv(DIRS["pathways"] / f"GO_{safe}_{direction}.csv",index=False); go_all.append(tab)
            ranking=d[["gene","stat"]].drop_duplicates("gene").sort_values("stat",ascending=False)
            if len(ranking)>=100:
                tab=gp.prerank(rnk=ranking,gene_sets=GO_LIBRARY,permutation_num=1000,seed=RANDOM_SEED,min_size=10,max_size=500,outdir=None,verbose=False).res2d
                tab["cell_type_broad"]=ct; tab.to_csv(DIRS["pathways"] / f"GSEA_GO_BP_{safe}.csv",index=False); gsea_all.append(tab)
    except Exception as exc: warnings.warn(f"Pathways não executados; DE e demais saídas permanecem válidas: {exc}")
if go_all: pd.concat(go_all,ignore_index=True).to_csv(DIRS["pathways"] / "GO_all_cell_types.csv",index=False)
if gsea_all: pd.concat(gsea_all,ignore_index=True).to_csv(DIRS["pathways"] / "GSEA_all_cell_types.csv",index=False)

## 9. Sensibilidade leave-one-animal-out (opcional)

In [ ]:
if RUN_DE_LOO:
    loo=[]
    for ct in de_results:
        full=de_results[ct].set_index("gene")
        for sample in SAMPLES:
            try:
                r,_,_=run_de(ct,sample); x=r.set_index("gene").join(full[["log2FoldChange","padj"]],rsuffix="_full")
                x["sign_concordant"]=np.sign(x.log2FoldChange)==np.sign(x.log2FoldChange_full); x["cell_type_broad"]=ct; x["left_out"]=sample; loo.append(x.reset_index())
            except Exception as exc: warnings.warn(f"LOO falhou {ct}/{sample}: {exc}")
    if loo: pd.concat(loo,ignore_index=True).to_csv(DIRS["de"] / "DE_leave_one_animal_out.csv",index=False)
else: print("LOO desativado. Ative apenas após revisar a análise primária.")

## 10. Workbook, manifesto e resumo automático

In [ ]:
workbook=OUTPUT_ROOT / "AdoTHC_analysis_summary.xlsx"
with pd.ExcelWriter(workbook,engine="openpyxl") as w:
    input_manifest.to_excel(w,"inputs",index=False); qc_summary.to_excel(w,"QC_samples",index=False)
    resolution_diagnostics.to_excel(w,"resolutions",index=False); annotation.to_excel(w,"annotation_broad",index=False)
    composition.to_excel(w,"composition",index=False); composition_stats.to_excel(w,"composition_stats",index=False)
    eligibility.to_excel(w,"PB_eligibility",index=False)
    if not de_all.empty: de_all.to_excel(w,"DE_all",index=False)

manifest={"design":"4 VEH vs 4 THC; independent mice; whole hippocampus; non-perfused","cellbender":"uniform lr=5e-5, filtered matrices",
          "parameters":{"MIN_GENES":MIN_GENES,"MIN_COUNTS":MIN_COUNTS,"MAX_PCT_MT":MAX_PCT_MT,"UPPER_MAD":UPPER_MAD,
                        "EXPECTED_DOUBLET_RATE":EXPECTED_DOUBLET_RATE,"N_TOP_HVG":N_TOP_HVG,"N_PCS":N_PCS,"N_NEIGHBORS":N_NEIGHBORS,
                        "BROAD_RESOLUTION":BROAD_RESOLUTION,"FINE_RESOLUTION":FINE_RESOLUTION,"USE_HARMONY":USE_HARMONY,
                        "MIN_NUCLEI_PER_SAMPLE_TYPE":MIN_NUCLEI_PER_SAMPLE_TYPE,"MIN_SAMPLES_PER_GROUP":MIN_SAMPLES_PER_GROUP,"seed":RANDOM_SEED},
          "versions":{p:version(p) for p in ["scanpy","anndata","pydeseq2","gseapy"]},
          "interpretation":["Animal is the replicate for THC vs VEH","Unannotated is retained when confidence is insufficient",
                            "Peripheral immune nuclei are biologically/technically plausible in non-perfused tissue","Effect sizes and uncertainty take priority over threshold crossing"]}
(DIRS["logs"] / "run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

summary=["# AdoTHC — resumo automático", "", f"- Núcleos finais: {adata.n_obs:,}", f"- Genes: {adata.n_vars:,}",
         f"- Clusters amplos: {adata.obs.cluster_broad.nunique()}", f"- Clusters detalhados: {adata.obs.cluster_fine.nunique()}",
         f"- Tipos amplos elegíveis para DE: {int(eligibility.eligible.sum())}", f"- Comparação: THC − VEH; n=4 animais/grupo", "",
         "## Alertas para revisão", "- Abra broad_marker_dotplot.png e cluster_annotation_probabilities.csv.",
         "- Examine fine_clusters_by_sample.csv para clusters dominados por uma amostra.", "- Examine post_CellBender_marker_leakage.csv antes de interpretar mielina/imunidade.",
         "- Confirme conclusões centrais com edgeR quasi-likelihood usando os arquivos edgeR_counts/metadata exportados."]
(DIRS["reports"] / "RESUMO_AUTOMATICO.md").write_text("\n".join(summary),encoding="utf-8")
adata.write_h5ad(DIRS["checkpoints"] / "04_final_annotated.h5ad",compression="gzip")
print("WORKFLOW CONCLUÍDO")
print("Workbook:",workbook)
print("Resumo:",DIRS["reports"] / "RESUMO_AUTOMATICO.md")
print("Objeto final:",DIRS["checkpoints"] / "04_final_annotated.h5ad")

## Ordem de revisão dos resultados

1. `qc/QC_distributions_before_filtering.png` e `qc_summary_by_sample.csv`.
2. `embeddings/UMAP_*`, `markers/resolution_diagnostics.csv` e dominância por amostra.
3. `annotation/broad_marker_dotplot.png` e as probabilidades; não force rótulos.
4. `composition/composition_animal_inference.csv` — pontos são animais.
5. `pseudobulk/pseudobulk_eligibility.csv` e PCA/contagens exportadas.
6. `de/DE_all_broad_cell_types.csv`, volcano plots e leave-one-out para conclusões centrais.
7. `pathways/` usando GSEA como análise principal de vias e ORA somente quando houver lista suficiente.

O notebook não remove automaticamente uma amostra inteira. Uma exclusão animal-level exige evidência convergente (QC, PCA pseudobulk, composição, influência LOO) e deve ser documentada antes de rerodar.